# STEP 1 - Model Initialization
1. Initial setup
2. Initialized model architecture
3. Load model weights


In [1]:
import torch

# step 1.1
# Set device (GPU if available)
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [2]:
# step 1.2
from model.attempt01.model import SimpleCNN

model = SimpleCNN().to(device)

In [3]:
# step 1.3
import glob
import os

# Find all .pth files and select the latest one by modified time
model_weight_files = glob.glob('./model/attempt01/*.safetensors')
if not model_weight_files:
    raise FileNotFoundError("No model weight files found in ./model/")
MODEL_WEIGHT_PATH = max(model_weight_files, key=os.path.getmtime)
print(f"Loading model weights from: {MODEL_WEIGHT_PATH}")

state_dict = torch.load(MODEL_WEIGHT_PATH, map_location=device)
model.load_state_dict(state_dict)



Loading model weights from: ./model/attempt01/simple_cnn_mnist_20261007_112945.safetensors


<All keys matched successfully>

# STEP 2 - Prepare Validation dataset

In [4]:
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST

# step 2.1

# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_dataset = MNIST(root='./data', download=True, train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


# STEP 3 - Forward Hook

In [5]:
activations = {}

def make_hook(name):
    def hook(module, inp, out):
        activations[name] = out.detach()
    return hook

handles = []
for name, module in model.named_children():
    handles.append(module.register_forward_hook(make_hook(name)))




In [ ]:
# step 2.2 - one validation image for inference
model.eval()

image, label = test_dataset[0]              # [1, 28, 28], int
image = image.unsqueeze(0).to(device)       # [1, 1, 28, 28]

with torch.no_grad():
    output = model(image)                   # log-probs [1, 10]
    # pred = output.argmax(dim=1).item()

# print(f"true={label}, pred={pred}")

true=7, pred=7


In [42]:
import torch.nn.functional as F

with torch.no_grad():
    print("captured layers:", list(activations.keys()))
    for name, t in activations.items():
        print(f"{name:10s}  shape={tuple(t.shape)}  mean={t.mean().item():.4f}  std={t.std().item():.4f}")


    # Print fc2 (final Linear layer) forward pass calculation output
    fc2_input = activations.get('dropout2')
    if fc2_input is not None:
        fc2_weight = model.fc2.weight
        fc2_bias = model.fc2.bias
        # Compute fc2 output manually, before log_softmax
        manual_fc2_out = torch.matmul(fc2_input, fc2_weight.T) + fc2_bias
        print("fc2 manual output (pre-log-softmax):", manual_fc2_out)
        print("fc2 manual output shape:", manual_fc2_out.shape)

        # calculate classification based on last layer output
        print(f"true={label}, predicted classification: {F.log_softmax(manual_fc2_out, dim=1).argmax(dim=1).item()}")

    else:
        print("fc1 activations not found in captured activations. Check hook names.")

# later: remove hooks
for h in handles:
    h.remove()

captured layers: ['conv1', 'conv2', 'dropout1', 'fc1', 'dropout2', 'fc2']
conv1       shape=(1, 32, 26, 26)  mean=-0.2095  std=0.4733
conv2       shape=(1, 64, 24, 24)  mean=-0.7058  std=0.7697
dropout1    shape=(1, 64, 12, 12)  mean=0.0410  std=0.1618
fc1         shape=(1, 128)  mean=-0.8073  std=3.0306
dropout2    shape=(1, 128)  mean=0.8400  std=1.4579
fc2         shape=(1, 10)  mean=-28.7037  std=12.9124
fc2 manual output (pre-log-softmax): tensor([[-36.5004, -29.2072, -25.9031, -25.8417, -32.8238, -37.3744, -43.8687,
           3.4475, -35.2463, -23.7187]], device='mps:0')
fc2 manual output shape: torch.Size([1, 10])
true=7, predicted classification: 7
